In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key=os.getenv("sample_qroq_api_key")


In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader('./attention.pdf')
docs=loader.load()
docs

[Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Tr

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=20)
final_documents=splitter.split_documents(docs)
final_documents

[Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Provided proper attribution is provided, Google hereby grants permission to'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='reproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='scholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='avaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Google Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗'),
 Document(metadata={'source': './attention.pdf', 'page': 0}, page_content='Llion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\n

In [ ]:
from langchain_community.embeddings.huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


KeyboardInterrupt: 

In [5]:
from langchain_chroma import Chroma

vector_db=Chroma.from_documents(embedding=embeddings,documents=final_documents)
vector_db

In [6]:
retriever=vector_db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x348198950>)

In [7]:
from langchain_groq import ChatGroq

import httpx


custom_http_client = httpx.Client(verify=False)
llm=ChatGroq(model="Llama3-8b-8192",groq_api_key=groq_key,http_client=custom_http_client)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8b-8192', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34b516650>)

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
        ("system",system_prompt),
        ("human","{input}")
    ]
)



prompt

ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])

In [11]:
# # from langchain_core.prompts import ChatPromptTemplate, SystemMessage, HumanMessage

# # Define the prompt template with input variables
# prompt = ChatPromptTemplate(
#     input_variables=["context", "input"],
#     messages=[
#         SystemMessage(content=(
#             "You are an assistant for question-answering tasks. "
#             "Use the following pieces of retrieved context to answer the question. "
#             "If you don't know the answer, say that you don't know. "
#             "Use three sentences maximum and keep the answer concise.\n\n{context}"
#         )),
#         HumanMessage(content="{input}")
#     ]
# )

# # Example values for the variables
# context_text = "The Eiffel Tower is located in Paris and was completed in 1889."
# question_text = "Where is the Eiffel Tower located?"

# # Format the prompt with those values
# formatted_prompt = prompt.format_prompt(context=context_text, input=question_text)

# # Print the resulting messages ready for the model
# for message in formatted_prompt.messages:
#     print(f"{message.type}: {message.content}")


# formatted_prompt


In [108]:
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough,RunnableMap

# method1

In [59]:
chain1={"context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),"input":RunnablePassthrough()}|prompt|llm
chain1

{
  context: RunnableLambda(lambda x: retriever.get_relevant_documents(x['input'])),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8b-8192', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34b516650>)

In [60]:
retriever.get_relevant_documents("what is encoder")

[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'),
 Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'),
 Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every'),
 Document(id='79551b47-bb91-46ec-9eee-6fc5573999b4', metadata={'page': 1, 'source': './attention.pdf'}, page_content='Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence')]

In [61]:
## will not fill values
#Because prompt.invoke() doesn’t run Runnables — it just substitutes values.


prompt.invoke({
    "context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),
    "input": "what is encoder"
})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nRunnableLambda(...)"), HumanMessage(content='what is encoder')])

In [62]:
prompt.invoke({
    "context": retriever.get_relevant_documents("what is encoder"),
    "input": "what is encoder"
})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'), Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'), Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every'), Document(id='79551b47-bb91-46ec-9eee-6fc5573999b4', metadata={'page': 1, 'source': './attention.pdf'}, page_content='Here, the encoder maps an input sequence of symbol 

In [63]:
chain_x={"context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),"input":RunnablePassthrough()}|prompt
chain_x.invoke({"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'), Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'), Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every'), Document(id='79551b47-bb91-46ec-9eee-6fc5573999b4', metadata={'page': 1, 'source': './attention.pdf'}, page_content='Here, the encoder maps an input sequence of symbol 

In [64]:
chain1.invoke({"input": "what is encoder"})

AIMessage(content='The encoder is a component in a neural network architecture that maps an input sequence of symbol representations to a sequence.', response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 307, 'total_tokens': 330, 'completion_time': 0.017978038, 'prompt_time': 0.071409295, 'queue_time': 1.952580269, 'total_time': 0.089387333}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-9ef0c0ca-df12-4200-8904-6e54f29b0e14-0', usage_metadata={'input_tokens': 307, 'output_tokens': 23, 'total_tokens': 330})

# method 2

In [65]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


context_retrieval_chain = (
    RunnableLambda(lambda x: x["input"]) |
    retriever |
    RunnableLambda(format_docs)
)

In [66]:
chain2={"context":context_retrieval_chain,"input":RunnablePassthrough()}|prompt|llm
chain2

{
  context: RunnableLambda(...)
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x348198950>)
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8b-8192', groq_api_key=SecretStr('********

In [67]:
context_retrieval_chain.invoke({"input": "what is encoder"})

'encoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every\n\nHere, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence'

In [68]:
prompt.invoke({
    "context": context_retrieval_chain.invoke({"input": "what is encoder"}),
    "input": "what is encoder"
})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every\n\nHere, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence"), HumanMessage(content='what is encoder')])

In [69]:
chain_x={"context":context_retrieval_chain,"input":RunnablePassthrough()}|prompt
chain_x.invoke({"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every\n\nHere, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence"), HumanMessage(content="{'input': 'what is encoder'}")])

In [70]:
chain2.invoke({"input": "what is encoder"})

AIMessage(content='The encoder is a component in a neural network that maps an input sequence of symbol representations to a sequence of dense representations, typically used in sequence-to-sequence tasks such as machine translation and language modeling.', response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 131, 'total_tokens': 172, 'completion_time': 0.032050562, 'prompt_time': 0.051433984, 'queue_time': 0.678988805, 'total_time': 0.083484546}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-7475e583-a11a-42b7-af3b-3912f0b4032e-0', usage_metadata={'input_tokens': 131, 'output_tokens': 41, 'total_tokens': 172})

# method3

In [90]:
chain3=RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])) 
    | RunnableLambda(format_docs),input=RunnablePassthrough()
)|prompt|llm
chain3

{
  context: RunnableLambda(lambda x: retriever.get_relevant_documents(x['input']))
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8b-8192', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34b516650>)

In [91]:
retriever.get_relevant_documents("what is encoder")

[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'),
 Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'),
 Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every'),
 Document(id='79551b47-bb91-46ec-9eee-6fc5573999b4', metadata={'page': 1, 'source': './attention.pdf'}, page_content='Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence')]

In [92]:
RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"]))
    | RunnableLambda(format_docs),input=RunnablePassthrough()
).invoke({"input": "what is encoder"})

{'context': 'encoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every\n\nHere, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence',
 'input': {'input': 'what is encoder'}}

In [93]:
# error
prompt.invoke(RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"]))
    | RunnableLambda(format_docs),input=RunnablePassthrough()
).invoke({"input": "what is encoder"}))

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every\n\nHere, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence"), HumanMessage(content="{'input': 'what is encoder'}")])

In [106]:
chain_x=RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"]))
    | RunnableLambda(format_docs),input=RunnablePassthrough()
)|prompt
chain_x.invoke({"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every\n\nHere, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence"), HumanMessage(content="{'input': 'what is encoder'}")])

In [95]:
chain3.invoke({"input": "what is encoder"})

AIMessage(content='The encoder is a component in a neural network that maps an input sequence of symbol representations to a sequence of vectors, typically used in transformer models and recurrent neural networks. It processes the input sequence and generates a sequence of outputs that can be used for subsequent tasks, such as machine translation or sentiment analysis.', response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 131, 'total_tokens': 192, 'completion_time': 0.048092946, 'prompt_time': 0.022938737, 'queue_time': 3.001328272, 'total_time': 0.071031683}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-fc5d7838-14b4-4862-957f-c8d5c450d28d-0', usage_metadata={'input_tokens': 131, 'output_tokens': 61, 'total_tokens': 192})

# method 4

In [116]:
retriever1=vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)
retriever1

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x348198950>, search_kwargs={'k': 3})

In [117]:
#LangChain internally wraps retriever1 so that when it receives {"input": "your query"}, 
# it automatically uses "input" as the query string — if no other fields are specified.


chain4={"context": retriever1,"input":RunnablePassthrough()}|prompt|llm
chain4

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x348198950>, search_kwargs={'k': 3}),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8b-8192', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34b516

In [118]:
retriever1.invoke("what is encoder")

[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'),
 Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'),
 Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every')]

In [119]:
prompt.invoke({"context":retriever1.invoke("what is encoder"),"input": "what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'), Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'), Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every')]"), HumanMessage(content='what is encoder')])

In [122]:
chain_x = {
    "context": retriever1,             # already a Runnable
    "input": RunnablePassthrough()     # passes the question as-is
}| prompt
chain_x.invoke("what is encoder")

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'), Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'), Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every')]"), HumanMessage(content='what is encoder')])

In [125]:
#this will raise error because 
# RunnableMap gives the entire input dict (i.e., {"input": "what is encoder"}) to both retriever1 and RunnablePassthrough().


# chain_x = {
#     "context": retriever1,             # already a Runnable
#     "input": RunnablePassthrough()     # passes the question as-is
# }| prompt
# chain_x.invoke({"input":"what is encoder"})

In [126]:
chain4.invoke("where is softmax used?")

AIMessage(content='The softmax function is used in attention mechanisms to calculate the weights of illegal connections.', response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 252, 'total_tokens': 269, 'completion_time': 0.013740248, 'prompt_time': 0.052402401, 'queue_time': 0.117941338, 'total_time': 0.066142649}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-b36c9107-4076-4219-9c85-1d1c10646e93-0', usage_metadata={'input_tokens': 252, 'output_tokens': 17, 'total_tokens': 269})

# method 5

In [127]:
retriever1.invoke("what is encoder")

[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'),
 Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'),
 Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every')]

In [ ]:
chain5={"context": RunnableLambda(lambda x:x["input"])|retriever1|RunnableLambda(format_docs),
        "input":RunnablePassthrough()
}|prompt|llm
chain5

{
  context: RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x348198950>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  input: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8

In [130]:
retriever1.invoke("what is encoder")

[Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'),
 Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'),
 Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every')]

In [132]:
format_docs(retriever1.invoke("what is encoder"))

'encoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every'

In [136]:
chain_x = {"context": RunnableLambda(lambda x:x["input"])|retriever1|RunnableLambda(format_docs),
        "input":RunnablePassthrough()
}|prompt
chain_x.invoke({"input":"what is encoder"})

ChatPromptValue(messages=[SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\nencoder.\n\nattention over the output of the encoder stack. Similar to the encoder, we employ residual\n\nand the memory keys and values come from the output of the encoder. This allows every"), HumanMessage(content="{'input': 'what is encoder'}")])

In [138]:
chain5.invoke({"input":"what is encoder"})

AIMessage(content='The encoder is a component in a neural network architecture that takes in input data and transforms it into a fixed-length representation. This representation is then used as input for the next component in the network, often the decoder.', response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 109, 'total_tokens': 153, 'completion_time': 0.036080227, 'prompt_time': 0.022430397, 'queue_time': 0.043137363, 'total_time': 0.058510624}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-ff5b1602-6706-4659-beff-b59f351ae449-0', usage_metadata={'input_tokens': 109, 'output_tokens': 44, 'total_tokens': 153})

# method6

In [139]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

stuffed_doc=create_stuff_documents_chain(prompt=prompt,llm=llm)
chain6=create_retrieval_chain(retriever,stuffed_doc)
print(stuffed_doc)
chain6

bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), config={'run_name': 'format_inputs'})
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8b-8192', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34b516650>)
| StrOutputParser() config={'run_name': 'stuff_documents_chain'}


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x348198950>), config={'run_name': 'retrieve_documents'})
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), config={'run_name': 'format_inputs'})
            | ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_v

In [220]:
chain6.invoke({"input":"what is encoder"})

{'input': 'what is encoder',
 'context': [Document(id='425d7d36-d46f-46e0-bfb0-3f1a58279b20', metadata={'page': 4, 'source': './attention.pdf'}, page_content='encoder.'),
  Document(id='324df977-6d20-4b7f-9017-81c7ca8cb50f', metadata={'page': 2, 'source': './attention.pdf'}, page_content='attention over the output of the encoder stack. Similar to the encoder, we employ residual'),
  Document(id='12929d2a-b2a6-4eed-b573-d134af75f727', metadata={'page': 4, 'source': './attention.pdf'}, page_content='and the memory keys and values come from the output of the encoder. This allows every'),
  Document(id='79551b47-bb91-46ec-9eee-6fc5573999b4', metadata={'page': 1, 'source': './attention.pdf'}, page_content='Here, the encoder maps an input sequence of symbol representations (x1, ..., xn) to a sequence')],
 'answer': 'In the context of sequence-to-sequence models, the encoder is a component that maps an input sequence of symbol representations to a fixed-length vector of outputs, typically a c

Great question — you're now using a more **abstracted and higher-level** way of building LangChain chains using:

* `create_stuff_documents_chain(...)`
* `create_retrieval_chain(...)`

These functions **hide a lot of internal wiring**, so understanding how the chain maps `"input"` and `"output"` is key to using it correctly.

---

## 🔍 Your Code Summary

```python
stuffed_doc = create_stuff_documents_chain(prompt=prompt, llm=llm)
chain6 = create_retrieval_chain(retriever, stuffed_doc)

response = chain6.invoke({"input": "where is softmax used?"})
```

---

## ✅ What Each Step Does

### 1. `create_stuff_documents_chain(prompt, llm)`

This creates a **Chain** where:

* The `retrieved documents` are **stuffed** (i.e., concatenated) into a prompt.
* The prompt is passed to the `llm`.

This chain expects:

```python
{"context": List[Document], "input": str}
```

LangChain will then:

* Format the prompt using both `{context}` and `{input}`.
* Pass it to the LLM.
* Return the LLM’s response.

📦 Output:

```python
{"output": "<llm_response>"}
```

---

### 2. `create_retrieval_chain(retriever, stuffed_doc)`

This wraps your `stuffed_doc` chain with a retriever.

It does the following steps internally:

1. Takes your input: `{"input": "where is softmax used?"}`
2. Runs `retriever.invoke("where is softmax used?")` to get documents.
3. Passes to `stuffed_doc`:

   ```python
   {
       "input": "where is softmax used?",
       "context": [Document1, Document2, ...]
   }
   ```
4. Receives the LLM output from `stuffed_doc`.
5. Returns a dict like:

   ```python
   {
       "input": "where is softmax used?",
       "context": [Document1, Document2, ...],
       "answer": "<llm response>"
   }
   ```

---

## 🎯 Final Mapping

Here’s a clear breakdown of what's happening in `chain6.invoke(...)`:

| Step                       | Input / Output                                             | Description                         |
| -------------------------- | ---------------------------------------------------------- | ----------------------------------- |
| `invoke({"input": "..."})` | Input dict with question                                   | Your initial input                  |
| `retriever`                | `"where is softmax used?"` → `[Document1, Document2, ...]` | Retrieves relevant context          |
| `stuffed_doc`              | `{input: ..., context: [...]}`                             | Builds prompt and sends to LLM      |
| `llm`                      | Generates final answer                                     | From stuffed prompt                 |
| `create_retrieval_chain`   | Returns full dict                                          | Includes input, context, and answer |

---

## 📥 Input Format

You must provide:

```python
{"input": "<your question>"}
```

## 📤 Output Format

You get back:

```python
{
    "input": "<your question>",
    "context": [<retrieved documents>],
    "answer": "<llm-generated answer>"
}
```

---

## 🔁 Example

```python
response = chain6.invoke({"input": "where is softmax used?"})
```

Sample output:

```python
{
    'input': 'where is softmax used?',
    'context': [Document(page_content="Softmax is used in neural networks..."), ...],
    'answer': 'Softmax is commonly used in the final layer of classification neural networks to convert logits into probabilities.'
}
```

You can access the result like:

```python
print(response["answer"])
```

---

## ✅ Summary

* `create_retrieval_chain()` automatically handles input mapping.
* It expects a key `"input"` and returns keys `"input"`, `"context"`, and `"answer"`.
* Internally, it maps `"input"` → retriever → `"context"` → prompt → LLM → `"answer"`.

---

Let me know if you want to:

* Customize the returned output (e.g. exclude `"context"`),
* Switch to a `map_reduce` or `refine` document chain instead of `stuff`,
* Use async for batching.


In [141]:
chain6.invoke({"input":"where is softmax used?"})

{'input': 'where is softmax used?',
 'context': [Document(id='3ce8b541-16a2-448d-92de-a8516065ab16', metadata={'page': 4, 'source': './attention.pdf'}, page_content='of the softmax which correspond to illegal connections. See Figure 2.'),
  Document(id='ff9315d7-39c5-4bb8-a61c-606f9c137542', metadata={'page': 3, 'source': './attention.pdf'}, page_content='dk, the dot products grow large in magnitude, pushing the softmax function into regions where it'),
  Document(id='527d59fa-bfe5-4d6b-8819-25ff15f0f3a3', metadata={'page': 4, 'source': './attention.pdf'}, page_content='dff = 2048.\n3.4 Embeddings and Softmax'),
  Document(id='3b3ec002-a02a-4358-af89-5d51577ac0b7', metadata={'page': 4, 'source': './attention.pdf'}, page_content='our model, we share the same weight matrix between the two embedding layers and the pre-softmax')],
 'answer': 'Softmax is used in the output layer of our model, specifically in the pre-softmax layer, where it is used to generate probability distributions over 

## adding history basic example

In [147]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(llm,get_session_history)
with_message_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableLambda(_enter_history), config={'run_name': 'load_history'})
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), config={'run_name': 'check_sync_or_async'}), config={'run_name': 'RunnableWithMessageHistory'}), get_session_history=<function get_session_history at 0x1503ed080>, history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [148]:
config1={"configurable":{"session_id":"chat1"}}

In [149]:
with_message_history.invoke(
    [
        HumanMessage(content="Hi,my name is anurag"),
        HumanMessage(content="what is my name?"),
    ],
    config=config1)

AIMessage(content="I'm happy to remind you! Your name is Anurag!", response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 33, 'total_tokens': 48, 'completion_time': 0.012716711, 'prompt_time': 0.004465794, 'queue_time': 0.046315476, 'total_time': 0.017182505}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-db2911b2-e6b3-4cab-bbf5-1b1e402f6761-0', usage_metadata={'input_tokens': 33, 'output_tokens': 15, 'total_tokens': 48})

In [150]:
config2={"configurable":{"session_id":"chat2"}}

In [151]:
with_message_history.invoke(
    [
        HumanMessage(content="what is my name?"),
    ],
    config=config2)

AIMessage(content="I'm sorry, but I don't know what your name is! As a conversational AI, I don't have any information about your personal identity or any previous interactions with you. Each time you interact with me, it's a new conversation and I don't retain any information from previous chats. Would you like to introduce yourself?", response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 15, 'total_tokens': 83, 'completion_time': 0.057010733, 'prompt_time': 0.003312753, 'queue_time': 0.106744456, 'total_time': 0.060323486}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-f2fe526c-c978-46a7-a092-9e5f4c00edf2-0', usage_metadata={'input_tokens': 15, 'output_tokens': 68, 'total_tokens': 83})

## adding history above implemntation

## method1

In [167]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)



prompt = ChatPromptTemplate.from_messages([
        ("system",system_prompt),
        ("human","{input}")
    ]
)

prompt

ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])

In [168]:
model1=prompt|llm
model1

ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise.\n\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x3a7e53cd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3a7f34610>, model_name='Llama3-8b-8192', groq_api_key=SecretStr('**********'), http_client=<httpx.Client object at 0x34b516650>)

In [158]:
store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]


In [174]:
with_message_history1=RunnableWithMessageHistory(model1,get_session_history,input_messages_key="input")
with_message_history1

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  input: RunnableBinding(bound=RunnableLambda(_enter_history), config={'run_name': 'load_history'})
}), config={'run_name': 'insert_history'})
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), config={'run_name': 'check_sync_or_async'}), config={'run_name': 'RunnableWithMessageHistory'}), get_session_history=<function get_session_history at 0x1503ed800>, input_messages_key='input', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [175]:
config={"configurable":{"session_id":"chat1"}}

In [177]:
#here context become history
with_message_history1.invoke({"context":"my name is anurag","input":"what is my name?"},config=config)

AIMessage(content='Your name is Anurag.', response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 459, 'total_tokens': 467, 'completion_time': 0.006867774, 'prompt_time': 0.052743814, 'queue_time': 0.048928965, 'total_time': 0.059611588}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-ea236b4f-2f1b-44b1-a572-9cf8e9a3c0c0-0', usage_metadata={'input_tokens': 459, 'output_tokens': 8, 'total_tokens': 467})

In [178]:
config={"configurable":{"session_id":"chat2"}}

In [182]:
# here retriver1  by default pass input query to it and pass context as history

# with_message_history1.invoke({"context":retriever1,"input":"where softmax function used?"},config=config)
with_message_history1.invoke({"context":retriever1|RunnableLambda(format_docs),"input":"where softmax function used?"},config=config)

AIMessage(content='The softmax function is used in the Hugging Face Embeddings in the Chroma vector store to calculate the probabilities of each token in the input sequence.', response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 995, 'total_tokens': 1026, 'completion_time': 0.02649946, 'prompt_time': 0.215502914, 'queue_time': 0.153106356, 'total_time': 0.242002374}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-2dfa2449-f32c-4bed-862f-6c8a8979f3f2-0', usage_metadata={'input_tokens': 995, 'output_tokens': 31, 'total_tokens': 1026})

In [183]:
store["chat2"]

InMemoryChatMessageHistory(messages=[HumanMessage(content='where softmax function used?'), AIMessage(content='The softmax function is used in the Hugging Face Embeddings in the Chroma vector store to calculate the probabilities of each token in the input sequence. This is typically done in natural language processing tasks, such as language modeling and text classification.', response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 115, 'total_tokens': 164, 'completion_time': 0.041154865, 'prompt_time': 0.013532677, 'queue_time': 0.048944743, 'total_time': 0.054687542}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-eed3105d-1c67-4052-a95c-bbc2a21f399d-0', usage_metadata={'input_tokens': 115, 'output_tokens': 49, 'total_tokens': 164}), HumanMessage(content='where softmax function used?'), AIMessage(content='The softmax function is used in the Hugging Face Embeddings in the Chroma vector stor

## importace of history_messages_key="history"

Got it! You're referring to the detailed parameters for **`RunnableWithMessageHistory`** in LangChain, especially the ones used to configure how history is passed in/out of the runnable. Let's break these down **with explanations and examples** so you can understand when and why to use each.

---

# RunnableWithMessageHistory: Parameters Explained

---

### 1. **`get_session_history`**

* **What:**
  A function that, given a session ID (string), returns an instance of `BaseChatMessageHistory` (or subclass) for that session.

* **Why:**
  This allows the runnable to **load and maintain the chat history for a specific user/session**. Each session has its own conversation history.

* **When to use:**
  When you want your runnable to support **multiple users or sessions**, each with their own stored message history.

* **Example:**

```python
from langchain.schema import BaseChatMessageHistory

def get_history(session_id: str) -> BaseChatMessageHistory:
    # Return the chat message history instance for this session_id
    return MyCustomChatMessageHistory.load(session_id)

runnable = RunnableWithMessageHistory(
    base_runnable=my_base_runnable,
    get_session_history=get_history,
    ...
)
```

---

### 2. **`input_messages_key`**

* **What:**
  If your base runnable takes a **dictionary** as input, this parameter tells the `RunnableWithMessageHistory` **which key in the input dict holds the new input messages** (list of messages or string).

* **Why:**
  To correctly extract the "current input" messages from a potentially larger input dict.

* **When to use:**
  When the input to your runnable is not a plain string but a dict containing multiple fields, including messages.

* **Example:**

Suppose your runnable expects input like:

```python
input_dict = {
    "user_id": "1234",
    "current_messages": [...],  # new input messages here
    "other_data": "foo"
}
```

You set:

```python
input_messages_key = "current_messages"
```

---

### 3. **`output_messages_key`**

* **What:**
  If your base runnable returns a **dictionary** as output, this tells which key in the output dict contains the output messages.

* **Why:**
  To correctly extract the generated messages from the runnable's output dict.

* **When to use:**
  When your runnable returns outputs in a dictionary, not just a string or list of messages.

* **Example:**

Runnable output might be:

```python
{
  "response_text": "Hello!",
  "messages": [...],  # output messages here
  "metadata": {...}
}
```

You set:

```python
output_messages_key = "messages"
```

---

### 4. **`history_messages_key`**

* **What:**
  If your base runnable **expects a separate key for the historical messages in the input dict**, specify this key.

* **Why:**
  To tell the runnable where to find past conversation history in the input.

* **When to use:**
  When your runnable’s input dict contains both new input messages *and* a separate key holding historical messages.

* **Example:**

Input dict might look like:

```python
{
  "new_messages": [...],
  "past_history": [...]  # historical messages here
}
```

You set:

```python
input_messages_key = "new_messages"
history_messages_key = "past_history"
```

---

### 5. **`history_factory_config`**

* **What:**
  Configuration dict for fields to pass to the chat history factory (`BaseChatMessageHistory` constructor). Controls things like max tokens, storage options, etc.

* **Why:**
  To customize how the chat history is created or loaded.

* **When to use:**
  When your chat history requires configuration, e.g., setting max length or special storage.

* **Example:**

```python
history_factory_config = {
    "max_length": 100,
    "storage_path": "/tmp/chat_histories"
}
```

---

# How These Work Together: Full Example

Let's say you have a base runnable that accepts a dict input like this:

```python
{
  "new_messages": [...],         # new input messages
  "past_messages": [...]         # history messages
}
```

and returns output like:

```python
{
  "result_text": "...",
  "generated_messages": [...]
}
```

You want to support per-session chat histories stored somewhere.

```python
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    # Load from some storage or DB by session_id
    return MyChatMessageHistory.load(session_id)

runnable_with_history = RunnableWithMessageHistory(
    base_runnable=my_base_runnable,
    get_session_history=get_session_history,
    input_messages_key="new_messages",
    history_messages_key="past_messages",
    output_messages_key="generated_messages",
    history_factory_config={
        "max_length": 50
    }
)
```

Now, when you run:

```python
response = runnable_with_history.invoke(
    {
      "session_id": "user123",
      "new_messages": [UserMessage(content="Hello!")],
      "past_messages": []
    }
)
```

* The runnable uses `get_session_history("user123")` to load the conversation history.
* It extracts new messages from `"new_messages"` key.
* It extracts past messages from `"past_messages"` key.
* It gets output messages from `"generated_messages"` key.

---

# Summary Table of Parameters

| Parameter                | Purpose                                       | Example Value                                   |
| ------------------------ | --------------------------------------------- | ----------------------------------------------- |
| `get_session_history`    | Function: session\_id → Chat history instance | `lambda session_id: MyHistory.load(session_id)` |
| `input_messages_key`     | Key for new input messages in input dict      | `"new_messages"`                                |
| `output_messages_key`    | Key for messages in output dict               | `"generated_messages"`                          |
| `history_messages_key`   | Key for history messages in input dict        | `"past_messages"`                               |
| `history_factory_config` | Config dict for history creation              | `{"max_length": 100}`                           |

---

If you'd like, I can help write a **minimal working example** or answer questions on how to integrate this in your specific LangChain use case. Would you want that?


In [189]:
#method1

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

chain1={"context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])),"input":RunnablePassthrough()}|prompt|llm

runnable = RunnableWithMessageHistory(
    chain1,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)


Answer 1: The encoder is a component of a neural network model that processes input data and generates a sequence of output values.
Answer 2: The decoder is a component of a neural network model that processes input data and generates a sequence of output values. It is composed of a stack of identical layers, including self-attention layers that allow each position in the decoder to attend to all positions in the input sequence.
Answer 3: Applications of attention in our model include self-attention mechanisms.


In [190]:
store

{'3b5c3a1a-41f0-4750-b77d-1257e1545e32': InMemoryChatMessageHistory(messages=[HumanMessage(content='What is encoder?'), AIMessage(content='The encoder is a component of a neural network model that processes input data and generates a sequence of output values.', response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 317, 'total_tokens': 340, 'completion_time': 0.01954254, 'prompt_time': 0.13489816, 'queue_time': 0.474683058, 'total_time': 0.1544407}, 'model_name': 'Llama3-8b-8192', 'system_fingerprint': 'fp_4b5fbf0ced', 'finish_reason': 'stop', 'logprobs': None}, id='run-ca749f2b-4545-4919-9e8e-09e4951b8d9d-0', usage_metadata={'input_tokens': 317, 'output_tokens': 23, 'total_tokens': 340}), HumanMessage(content='What is decoder?'), AIMessage(content='The decoder is a component of a neural network model that processes input data and generates a sequence of output values. It is composed of a stack of identical layers, including self-attention layers that allow each 

In [191]:
#method2

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


context_retrieval_chain = (
    RunnableLambda(lambda x: x["input"]) |
    retriever |
    RunnableLambda(format_docs)
)
chain2={"context":context_retrieval_chain,"input":RunnablePassthrough()}|prompt|llm
chain2

runnable = RunnableWithMessageHistory(
    chain2,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)



Answer 1: The encoder is a component in a neural network that processes input sequences and transforms them into a fixed-length representation. It is typically composed of a stack of identical layers, each consisting of an encoder layer and a residual connection. The output of the encoder is used to compute attention over the input sequence.
Answer 2: The decoder is also composed of a stack of identical layers, similar to the encoder, and is used to process the output of the encoder. It allows each position in the decoder to attend over all positions in the input sequence. This mimics the way humans process language by considering the context of the entire sentence.
Answer 3: I don't know. The context provided does not directly answer the question about the applications of attention.


In [193]:
#method3

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

chain3=RunnableParallel(
    context=RunnableLambda(lambda x: retriever.get_relevant_documents(x["input"])) 
    | RunnableLambda(format_docs),input=RunnablePassthrough()
)|prompt|llm

runnable = RunnableWithMessageHistory(
    chain3,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)



Answer 1: The encoder is a component in a neural network architecture, typically used in natural language processing tasks, such as machine translation and text summarization. It takes in a sequence of input tokens and generates a continuous representation of the input sequence.
Answer 2: The decoder is also a component in a neural network architecture, similar to the encoder. It is composed of a stack of identical layers and allows each position in the decoder to attend to all positions in the input sequence.
Answer 3: I don't know.


In [215]:
# #method4

# #this will fail always

# import uuid

# system_prompt = (
#     "You are an assistant for question-answering tasks. "
#     "Use the following pieces of retrieved context to answer "
#     "the question. If you don't know the answer, say that you "
#     "don't know. Use three sentences maximum and keep the "
#     "answer concise."
#     "\n\n"
#     "{context}"
# )

# prompt = ChatPromptTemplate.from_messages([
#     ("system", system_prompt),
#     ("human", "{input}")
# ])



# # 4. Create dummy documents to simulate a retriever
# from langchain.docstore.document import Document

# docs = [
#     Document(page_content="LangChain is a framework for building LLM-powered applications."),
#     Document(page_content="FAISS is a library for efficient similarity search."),
#     Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
# ]



# store={}
# def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
#     if session_id not in store:
#         store[session_id] = ChatMessageHistory()

    
#     return store[session_id]

# # 7. Combine all using RunnableWithMessageHistory
# #    Use retriever to get context, inject into prompt, then pass to model
# # chain = (
# #     {
# #         "context": lambda x: retriever.get_relevant_documents(x["input"]),
# #         "input": lambda x: x["input"]
# #     }
# #     | prompt
# #     | llm
# # )

# retriever1=vector_db.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k":3}
# )



# chain4={"context": retriever1,"input":RunnablePassthrough()}|prompt|llm
# chain4

# runnable = RunnableWithMessageHistory(
#     chain4,
#     # lambda session_id: get_session_history1(session_id),
#     get_session_history,
#     input_messages_key="input",
#     # history_messages_key="history"
# )

# # 8. Use it in a session
# session_id = str(uuid.uuid4())  # Generate unique session ID

# # Ask first question
# response1 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
# print("Answer 1:", response1.content)

# # Ask second question (with context retained)
# response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
# print("Answer 2:", response2.content)

# # Ask third question (could be unrelated or test memory/context)
# response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
# print("Answer 3:", response3.content)



In [216]:
#method5

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

chain5={"context": RunnableLambda(lambda x:x["input"])|retriever1|RunnableLambda(format_docs),
        "input":RunnablePassthrough()
}|prompt|llm
chain5

runnable = RunnableWithMessageHistory(
    chain5,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1.content)

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2.content)

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3.content)



Answer 1: The encoder is a component in a neural network that converts input data into a fixed-length representation, known as a context vector. This is typically done using a multi-layer transformer model. The output of the encoder is then used as input to the next layer, such as the decoder.
Answer 2: The decoder is a component in a neural network that converts the output of the encoder into a target sequence. It's also composed of a stack of identical layers, including self-attention layers.
Answer 3: The applications of attention in our model include such attention mechanisms, allowing the model to focus on specific parts of the input sequence when generating the output sequence. This is particularly useful in tasks like machine translation, where the model needs to consider the context of the input sentence when generating the output sentence.


In [219]:
#method6

import uuid

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])



# 4. Create dummy documents to simulate a retriever
from langchain.docstore.document import Document

docs = [
    Document(page_content="LangChain is a framework for building LLM-powered applications."),
    Document(page_content="FAISS is a library for efficient similarity search."),
    Document(page_content="OpenAI offers powerful models like GPT-4 for text generation."),
]



store={}
def get_session_history(session_id: str, context=None) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    
    return store[session_id]

# 7. Combine all using RunnableWithMessageHistory
#    Use retriever to get context, inject into prompt, then pass to model
# chain = (
#     {
#         "context": lambda x: retriever.get_relevant_documents(x["input"]),
#         "input": lambda x: x["input"]
#     }
#     | prompt
#     | llm
# )

stuffed_doc=create_stuff_documents_chain(prompt=prompt,llm=llm)
chain6=create_retrieval_chain(retriever,stuffed_doc)

runnable = RunnableWithMessageHistory(
    chain6,
    # lambda session_id: get_session_history1(session_id),
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    output_messages_key="answer"
)

# 8. Use it in a session
session_id = str(uuid.uuid4())  # Generate unique session ID

# Ask first question
response1 = runnable.invoke({"input": "What is encoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 1:", response1['answer'])

# Ask second question (with context retained)
response2 = runnable.invoke({"input": "What is decoder?"}, config={"configurable": {"session_id": session_id}})
print("Answer 2:", response2['answer'])

# Ask third question (could be unrelated or test memory/context)
response3 = runnable.invoke({"input": "What are Applications of Attention?"}, config={"configurable": {"session_id": session_id}})
print("Answer 3:", response3['answer'])



Answer 1: The encoder is a component of a neural network model, composed of a stack of identical layers, used to process input data. Each layer of the encoder has two components: an encoder and residual connections, allowing it to learn complex patterns in the input data. The output of the encoder is used to calculate attention over its own output.
Answer 2: The decoder is composed of a stack of 6 identical layers, in addition to the two encoder layers. It allows each position in the decoder to attend to all positions in the input sequence, mimicking human-like language translation.
Answer 3: According to the given context, Applications of Attention in our Model refer to the usage of attention mechanisms in our model, specifically in sections 3.2.3.
